In [2]:
import sklearn
import numpy as np
import pandas as pd
import os
import re 
import sys 
import importlib

# Get the absolute path of the project root
project_root = os.getcwd()
# Define data directory
brighten_dir = os.path.join(project_root, 'BRIGHTEN_data')
sub_dir = os.path.join(project_root, 'BRIGHTEN_data', 'sub_dfs')
results_dir = os.path.join(project_root, 'results')


# Add project root to sys.path for script usage
sys.path.append(project_root)

# Import and reload (optional) custom scripts
from scripts import preprocessing as pre
from scripts import visualization as vis
from scripts import variables 
from scripts import feature_selection as fs
importlib.reload(pre)
importlib.reload(vis)
importlib.reload(variables)
importlib.reload(fs)


# Define label variables
df_names = ['v1_day', 'v2_day', 'v1_week', 'v2_week']




In [ ]:
# Create binary of depression score at the end of the time
count=0
start_depressed=np.nan
end_depressed=np.nan
last_phq9=np.nan
change_bin=np.nan
for name in ['v1_day','v2_day']: 
	phq9_end = []
	Xy = pd.read_csv(os.path.join(brighten_dir, f"{name}_weekFilled.csv"))
	for sub, sub_df in Xy.groupby('num_id'):
		sub_phq9 = sub_df.dropna(subset='phq9_sum')
		if len(sub_phq9) == 0:
			continue
		sub_phq9 = sub_phq9.sort_values(by='day', ascending=True)

		first_phq9 = list(sub_phq9['phq9_sum'])[0]
		if first_phq9 > 12:
			start_depressed = 1
		else:
			start_depressed = 0


		# Phq9 at ~6 weeks
		days6weeks=sub_phq9[sub_phq9['day']>38]
		days6weeks=days6weeks[days6weeks['day']<55]
		if not len(days6weeks) > 0:
			phq9_end.append([sub, first_phq9, start_depressed, last_phq9, end_depressed, change_bin]) #np.nan for last_phq9, end_depressed
			continue
		days6weeks_cols = days6weeks.dropna(how='all', axis=1)
		if not 'phq9_sum' in days6weeks_cols:
			phq9_end.append([sub, first_phq9, start_depressed, last_phq9, end_depressed, change_bin]) #np.nan for last_phq9, end_depressed
			continue

		last_phq9 = list(days6weeks['phq9_sum'])[0]
		if last_phq9 > 12:
			end_depressed = 1
		else:
			end_depressed = 0
		
		change_bin = start_depressed - end_depressed

		# if count < 3:
		# 	count+=1
		# 	print(f'Sub: {sub}')
		# 	display(days6weeks[['day','dt','phq9_sum']])
		# 	display(f'day: {list(days6weeks['day'])[0]}, phq9: {list(days6weeks['phq9_sum'])[0]}')
			

		phq9_end.append([sub, first_phq9, start_depressed, last_phq9, end_depressed, change_bin])

	phq9_end_df = pd.DataFrame(phq9_end, columns=['num_id', 'phq9_sum_start', 'start_depressed_binary', 'phq9_sum_6wks', '6wks_depressed_binary', 'depression_change_bin'])

	phq9_end_df.to_csv(os.path.join(brighten_dir, f'{name}_phq9sum_6wks.csv'), index=False)
	display(phq9_end_df)
	print(f'Saved phq9_end_df to {name}_phq9sum_6wks.csv')




,num_id,phq9_sum_start,start_depressed_binary,phq9_sum_6wks,6wks_depressed_binary,depression_change_bin
0,13.0,18.0,1,9.0,0,1
1,14.0,10.0,0,12.0,0,0
2,17.0,8.0,0,18.0,1,-1
3,22.0,12.0,0,18.0,1,-1
4,25.0,12.0,0,18.0,1,-1
...,...,...,...,...,...,...
269,1908.0,18.0,1,17.0,1,0
270,1911.0,22.0,1,11.0,0,1
271,1912.0,4.0,0,6.0,0,0
272,1913.0,5.0,0,6.0,0,0


Saved phq9_end_df to v1_day_phq9sum_6wks.csv


/var/folders/fl/b24z_8kn4490x_bl0njv6fg00000gn/T/ipykernel_51359/205975929.py:9: DtypeWarning: Columns (87) have mixed types. Specify dtype option on import or set low_memory=False.
  Xy = pd.read_csv(os.path.join(brighten_dir, f"{name}_weekFilled.csv"))


,num_id,phq9_sum_start,start_depressed_binary,phq9_sum_6wks,6wks_depressed_binary,depression_change_bin
0,161.0,9.0,0,3.0,0,0
1,166.0,14.0,1,12.0,0,1
2,174.0,15.0,1,12.0,0,1
3,175.0,16.0,1,12.0,0,1
4,176.0,17.0,1,12.0,0,1
...,...,...,...,...,...,...
214,1053.0,18.0,1,4.0,0,1
215,1075.0,5.0,0,4.0,0,1
216,1081.0,19.0,1,13.0,1,0
217,1096.0,9.0,0,8.0,0,0


Saved phq9_end_df to v2_day_phq9sum_6wks.csv


In [ ]:
v1_phq9_end_df=pd.read_csv(os.path.join(brighten_dir, f'v1_day_phq9sum_6wks.csv'))
v2_phq9_end_df=pd.read_csv(os.path.join(brighten_dir, f'v2_day_phq9sum_6wks.csv'))
phq9_end_df = pd.concat([v1_phq9_end_df, v2_phq9_end_df], axis=0)
phq9_end_df = phq9_end_df.loc[:, ~phq9_end_df.columns.str.contains('Unnamed')]
display(phq9_end_df)

phq9_end_df.to_csv(os.path.join(brighten_dir, f'phq9sum_6wks.csv'), index=False)

,num_id,phq9_sum_start,start_depressed_binary,phq9_sum_6wks,6wks_depressed_binary,depression_change_bin
0,13.0,18.0,1,9.0,0,1
1,14.0,10.0,0,12.0,0,0
2,17.0,8.0,0,18.0,1,-1
3,22.0,12.0,0,18.0,1,-1
4,25.0,12.0,0,18.0,1,-1
...,...,...,...,...,...,...
214,1053.0,18.0,1,4.0,0,1
215,1075.0,5.0,0,4.0,0,1
216,1081.0,19.0,1,13.0,1,0
217,1096.0,9.0,0,8.0,0,0
